# Bank Marketing Campaign Analysis — DA-014

This notebook is self-contained for Google Colab. Upload this notebook and `bank-additional-full.csv` (keep that exact filename) to the Colab runtime, then run all cells. The CSV may be in the runtime root or in a `data/` folder; no separate `analysis.py` file is required.

The embedded cleaning and calculation functions match the project dashboard's analysis logic. The target is `y` (yes/no), and response rate uses valid contact records as the denominator.


In [ ]:
"""Shared data loading and analysis functions for project DA-014.

Both the Streamlit dashboard and the Jupyter notebook use these functions so
that the response-rate denominator and group calculations stay consistent.
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Mapping

import numpy as np
import pandas as pd
import plotly.express as px


PROJECT_DIR = Path.cwd()
DATA_CANDIDATES = [
    PROJECT_DIR / "bank-additional-full.csv",
    PROJECT_DIR / "data" / "bank-additional-full.csv",
]
DATA_PATH = next((candidate for candidate in DATA_CANDIDATES if candidate.is_file()), DATA_CANDIDATES[0])
TARGET_COLUMN = "y"

CATEGORICAL_COLUMNS = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

NUMERIC_COLUMNS = [
    "age",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

SEGMENT_COLUMNS = CATEGORICAL_COLUMNS
CAMPAIGN_BUCKETS = ["1", "2", "3", "4", "5+"]
MONTH_ORDER = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
MIN_GROUP_RECORDS = 30


def clean_data(raw: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, object]]:
    """Normalize the UCI file without discarding valid contact records.

    The UCI file is semicolon-delimited. Literal ``unknown`` values are kept as
    an explicit category, ``?`` is converted to a missing value, numeric fields
    are parsed, and rows without a valid yes/no target are excluded from the
    response-rate denominator.
    """
    if raw.empty:
        raise ValueError("The dataset contains no records.")

    df = raw.copy()
    df.columns = [str(column).strip() for column in df.columns]
    if TARGET_COLUMN not in df.columns:
        raise ValueError(f"Required target column '{TARGET_COLUMN}' was not found.")

    raw_record_count = int(len(df))
    text_columns = df.select_dtypes(include=["object", "string"]).columns
    for column in text_columns:
        values = df[column].astype("string").str.strip()
        values = values.replace({"?": pd.NA})
        if column != TARGET_COLUMN:
            values = values.mask(values.str.lower().eq("unknown"), "Unknown")
        df[column] = values

    for column in NUMERIC_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    target = df[TARGET_COLUMN].astype("string").str.strip().str.lower()
    valid_target = target.isin(["yes", "no"])
    excluded_target_rows = int((~valid_target).sum())
    df[TARGET_COLUMN] = target
    df = df.loc[valid_target].reset_index(drop=True)

    if df.empty:
        raise ValueError("No rows with a valid yes/no target remain after cleaning.")

    unknown_counts = {
        column: int(df[column].eq("Unknown").sum())
        for column in CATEGORICAL_COLUMNS
        if column in df.columns
    }
    quality = {
        "raw_records": raw_record_count,
        "retained_records": int(len(df)),
        "excluded_invalid_target_records": excluded_target_rows,
        "missing_cells_after_cleaning": int(df.isna().sum().sum()),
        "unknown_category_counts": unknown_counts,
    }
    return df, quality


def load_data(path: str | Path = DATA_PATH) -> tuple[pd.DataFrame, dict[str, object]]:
    """Read and clean the official semicolon-delimited UCI CSV."""
    data_path = Path(path)
    if not data_path.exists():
        raise FileNotFoundError(
            "CSV not found. Upload bank-additional-full.csv to Colab Files "
            "or place it in the data/ folder."
        )
    raw = pd.read_csv(data_path, sep=";", skipinitialspace=True)
    return clean_data(raw)


def calculate_kpis(data: pd.DataFrame) -> dict[str, int | float]:
    """Calculate the headline metrics for the currently selected records.

    Response rate = records with y == 'yes' / records with valid y in the
    current cohort. Rows represent campaign contact records, not unique people.
    """
    records = int(len(data))
    responses = int(data[TARGET_COLUMN].eq("yes").sum()) if records else 0
    rate = responses / records if records else float("nan")
    average_contacts = (
        float(data["campaign"].mean())
        if records and "campaign" in data.columns
        else float("nan")
    )
    return {
        "records": records,
        "responses": responses,
        "response_rate": rate,
        "average_campaign_contacts": average_contacts,
    }


def apply_filters(
    data: pd.DataFrame,
    categorical_filters: Mapping[str, Iterable[object]] | None = None,
    numeric_ranges: Mapping[str, tuple[float, float]] | None = None,
) -> pd.DataFrame:
    """Apply sidebar filters and return a copy of the selected cohort."""
    filtered = data
    for column, selected_values in (categorical_filters or {}).items():
        if column not in filtered.columns:
            continue
        selected = list(selected_values)
        filtered = filtered.loc[filtered[column].isin(selected)]

    for column, bounds in (numeric_ranges or {}).items():
        if column not in filtered.columns:
            continue
        lower, upper = bounds
        filtered = filtered.loc[filtered[column].between(lower, upper, inclusive="both")]

    return filtered.copy()


def response_by_segment(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Return record counts, successful responses, and response rates by group."""
    if column not in data.columns:
        raise KeyError(f"Column '{column}' is not in the current dataset.")
    if data.empty:
        return pd.DataFrame(columns=["segment", "records", "responses", "response_rate"])

    table = (
        data.groupby(column, dropna=False, observed=True)[TARGET_COLUMN]
        .agg(records="size", responses=lambda values: int(values.eq("yes").sum()))
        .reset_index()
        .rename(columns={column: "segment"})
    )
    table["response_rate"] = table["responses"] / table["records"]
    table["segment"] = table["segment"].astype("string").fillna("Missing")
    return table.sort_values(["response_rate", "records"], ascending=[False, False]).reset_index(drop=True)


def contact_frequency_table(data: pd.DataFrame) -> pd.DataFrame:
    """Summarize outcomes by campaign contact-frequency bands.

    The UCI ``campaign`` field is the number of contacts to that record/client
    during this campaign, including the last contact.
    """
    if data.empty or "campaign" not in data.columns:
        return pd.DataFrame(columns=["contact_band", "records", "responses", "response_rate"])

    campaign = pd.to_numeric(data["campaign"], errors="coerce")
    bands = pd.Series(
        np.select(
            [campaign.eq(1), campaign.eq(2), campaign.eq(3), campaign.eq(4), campaign.ge(5)],
            CAMPAIGN_BUCKETS,
            default="Missing/other",
        ),
        index=data.index,
        dtype="string",
    )
    temporary = data.assign(_contact_band=bands)
    table = (
        temporary.groupby("_contact_band", observed=True)[TARGET_COLUMN]
        .agg(records="size", responses=lambda values: int(values.eq("yes").sum()))
        .reset_index()
        .rename(columns={"_contact_band": "contact_band"})
    )
    table["response_rate"] = table["responses"] / table["records"]
    order = {label: index for index, label in enumerate(CAMPAIGN_BUCKETS + ["Missing/other"])}
    table["_order"] = table["contact_band"].map(order)
    return table.sort_values("_order").drop(columns="_order").reset_index(drop=True)


def generate_findings(
    data: pd.DataFrame,
    segment_column: str = "job",
    min_group_records: int = MIN_GROUP_RECORDS,
) -> list[dict[str, str]]:
    """Create descriptive, sample-size-aware findings for the selected cohort."""
    if data.empty:
        return [{
            "title": "No records in the current selection",
            "finding": "Adjust or clear filters to calculate findings.",
            "recommendation": "Review a broader cohort before making campaign decisions.",
        }]

    findings: list[dict[str, str]] = []
    segments = response_by_segment(data, segment_column)
    eligible = segments.loc[segments["records"] >= min_group_records]
    if not eligible.empty:
        top = eligible.iloc[0]
        findings.append({
            "title": f"Highest observed response: {segment_column.replace('_', ' ').title()}",
            "finding": (
                f"{top['segment']} has the highest observed rate in the selected cohort: "
                f"{top['responses']:,} of {top['records']:,} records "
                f"({top['response_rate']:.1%}). Groups below {min_group_records} records are omitted."
            ),
            "recommendation": (
                "Use this segment as a hypothesis for a targeted campaign test; this descriptive "
                "association does not establish that the segment caused the response."
            ),
        })
    else:
        findings.append({
            "title": "Segment sample is too small",
            "finding": (
                f"No {segment_column.replace('_', ' ')} group has at least "
                f"{min_group_records} records in the current selection."
            ),
            "recommendation": "Broaden the filters before comparing segment response rates.",
        })

    frequency = contact_frequency_table(data).set_index("contact_band")
    one = frequency.loc["1"] if "1" in frequency.index else None
    repeated = frequency.loc["5+"] if "5+" in frequency.index else None
    if (
        one is not None
        and repeated is not None
        and one["records"] >= min_group_records
        and repeated["records"] >= min_group_records
    ):
        higher_band = "5+ contacts" if repeated["response_rate"] > one["response_rate"] else "one contact"
        difference_pp = abs(repeated["response_rate"] - one["response_rate"]) * 100
        direction = "higher" if repeated["response_rate"] > one["response_rate"] else "lower"
        findings.append({
            "title": "Contact-frequency comparison",
            "finding": (
                f"Response was {one['response_rate']:.1%} after one contact "
                f"({int(one['records']):,} records) and {repeated['response_rate']:.1%} after "
                f"5+ contacts ({int(repeated['records']):,} records); the 5+ band was "
                f"{difference_pp:.1f} percentage points {direction}."
            ),
            "recommendation": (
                f"Review the contact cadence for the {higher_band} group before changing follow-up "
                "rules; the comparison is observational and may reflect differences in who was contacted."
            ),
        })
    else:
        findings.append({
            "title": "Contact-frequency comparison needs more records",
            "finding": (
                f"The one-contact and 5+ contact bands do not both meet the "
                f"{min_group_records}-record display threshold in the current selection."
            ),
            "recommendation": "Use the chart to inspect the available contact bands, but avoid a cadence recommendation from small groups.",
        })

    if "contact" in data.columns:
        channels = response_by_segment(data, "contact")
        channels = channels.loc[channels["records"] >= min_group_records]
        if len(channels) >= 2:
            best = channels.iloc[0]
            other = channels.iloc[-1]
            findings.append({
                "title": "Communication-channel comparison",
                "finding": (
                    f"{best['segment']} has the highest observed channel rate at {best['response_rate']:.1%} "
                    f"({int(best['responses']):,}/{int(best['records']):,}); the comparison channel "
                    f"{other['segment']} is {other['response_rate']:.1%} "
                    f"({int(other['responses']):,}/{int(other['records']):,})."
                ),
                "recommendation": "Use channel differences to design a controlled follow-up test; customer mix and campaign timing can also explain the gap.",
            })
        else:
            findings.append({
                "title": "Channel comparison needs more records",
                "finding": f"Fewer than two communication channels meet the {min_group_records}-record threshold.",
                "recommendation": "Review a larger cohort before recommending a channel shift.",
            })

    return findings


def quality_table(quality: Mapping[str, object]) -> pd.DataFrame:
    """Turn cleaning metadata into a simple table for notebook display."""
    unknown_counts = quality.get("unknown_category_counts", {})
    if not isinstance(unknown_counts, Mapping):
        unknown_counts = {}
    rows = [
        ("Rows read from CSV", quality.get("raw_records", 0)),
        ("Rows retained with valid target", quality.get("retained_records", 0)),
        ("Rows excluded for invalid target", quality.get("excluded_invalid_target_records", 0)),
        ("Missing cells after cleaning", quality.get("missing_cells_after_cleaning", 0)),
        ("Explicit Unknown categories", int(sum(unknown_counts.values()))),
    ]
    return pd.DataFrame(rows, columns=["Data-quality measure", "Value"])


# Load the uploaded CSV; it can be beside this notebook or inside data/.
data, quality = load_data()
print(f"Loaded {len(data):,} valid contact records and {data.shape[1]} columns from {DATA_PATH}.")


## 1. Cleaning and data quality

The cleaning rules preserve `Unknown` categories, convert `?` to missing, parse numeric fields, and remove only rows with an invalid target.

In [2]:
quality_table(quality)

,Data-quality measure,Value
0,Rows read from CSV,41188
1,Rows retained with valid target,41188
2,Rows excluded for invalid target,0
3,Missing cells after cleaning,0
4,Explicit Unknown categories,12718


In [3]:
display(data.head())
display(data.dtypes.rename('dtype').to_frame())
display(data['y'].value_counts(dropna=False).rename_axis('outcome').to_frame('records'))

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,Unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


,dtype
age,int64
job,string
marital,string
education,string
default,string
housing,string
loan,string
contact,string
month,string
day_of_week,string


,records
outcome,
no,36548
yes,4640


## 2. Overall campaign response rate

The denominator includes all retained records with a valid target. The rate is computed directly from `y`; no target balance or KPI is hard-coded.

In [4]:
overall = calculate_kpis(data)
pd.DataFrame([
    {'Metric': 'Records analyzed', 'Value': f"{overall['records']:,}"},
    {'Metric': 'Successful responses (y = yes)', 'Value': f"{overall['responses']:,}"},
    {'Metric': 'Response rate', 'Value': f"{overall['response_rate']:.2%}"},
    {'Metric': 'Average contacts during this campaign', 'Value': f"{overall['average_campaign_contacts']:.2f}"},
])

,Metric,Value
0,Records analyzed,"41,188"
1,Successful responses (y = yes),"4,640"
2,Response rate,11.27%
3,Average contacts during this campaign,2.57


## 3. Response rates across customer and campaign segments

Each result includes response counts and record denominators so percentages can be checked. Small groups should be interpreted cautiously.

In [5]:
segment_tables = {}
for column in CATEGORICAL_COLUMNS:
    table = response_by_segment(data, column)
    segment_tables[column] = table
    print(f'\nResponse by {column}:')
    display(table.style.format({'response_rate': '{:.2%}'}))


Response by job:

,segment,records,responses,response_rate
0,student,875,275,31.43%
1,retired,1720,434,25.23%
2,unemployed,1014,144,14.20%
3,admin.,10422,1352,12.97%
4,management,2924,328,11.22%
5,Unknown,330,37,11.21%
6,technician,6743,730,10.83%
7,self-employed,1421,149,10.49%
8,housemaid,1060,106,10.00%
9,entrepreneur,1456,124,8.52%



Response by marital:


,segment,records,responses,response_rate
0,Unknown,80,12,15.00%
1,single,11568,1620,14.00%
2,divorced,4612,476,10.32%
3,married,24928,2532,10.16%



Response by education:


,segment,records,responses,response_rate
0,illiterate,18,4,22.22%
1,Unknown,1731,251,14.50%
2,university.degree,12168,1670,13.72%
3,professional.course,5243,595,11.35%
4,high.school,9515,1031,10.84%
5,basic.4y,4176,428,10.25%
6,basic.6y,2292,188,8.20%
7,basic.9y,6045,473,7.82%


... (10 outputs left out)


Response by day_of_week:


,segment,records,responses,response_rate
0,thu,8623,1045,12.12%
1,tue,8090,953,11.78%
2,wed,8134,949,11.67%
3,fri,7827,846,10.81%
4,mon,8514,847,9.95%



Response by poutcome:


,segment,records,responses,response_rate
0,success,1373,894,65.11%
1,failure,4252,605,14.23%
2,nonexistent,35563,3141,8.83%


In [6]:
job_rates = response_by_segment(data, 'job').sort_values('response_rate')
fig = px.bar(job_rates, x='response_rate', y='segment', orientation='h', hover_data=['records', 'responses'], title='Term-deposit response rate by job')
fig.update_xaxes(tickformat='.0%', title='Response rate')
fig.show()

## 4. Contact frequency, channel, and timing

`campaign` is the number of contacts during this campaign, including the last contact. Channel, month, and weekday comparisons are descriptive and can be affected by customer mix and timing.

In [7]:
frequency = contact_frequency_table(data)
display(frequency.style.format({'response_rate': '{:.2%}'}))
fig = px.bar(frequency, x='contact_band', y='response_rate', hover_data=['records', 'responses'], title='Response rate by number of campaign contacts')
fig.update_yaxes(tickformat='.0%', title='Response rate')
fig.show()

for column in ['contact', 'month', 'day_of_week']:
    table = response_by_segment(data, column)
    label = column.replace('_', ' ')
    print(f'\n{label.title()} comparison:')
    display(table.style.format({'response_rate': '{:.2%}'}))
    fig = px.bar(table, x='segment', y='response_rate', hover_data=['records', 'responses'], title=f'Response rate by {label}')
    fig.update_yaxes(tickformat='.0%', title='Response rate')
    fig.show()

,contact_band,records,responses,response_rate
0,1,17642,2300,13.04%
1,2,10570,1211,11.46%
2,3,5341,574,10.75%
3,4,2651,249,9.39%
4,5+,4984,306,6.14%



Contact comparison:


,segment,records,responses,response_rate
0,cellular,26144,3853,14.74%
1,telephone,15044,787,5.23%



Month comparison:


,segment,records,responses,response_rate
0,mar,546,276,50.55%
1,dec,182,89,48.90%
2,sep,570,256,44.91%
3,oct,718,315,43.87%
4,apr,2632,539,20.48%
5,aug,6178,655,10.60%
6,jun,5318,559,10.51%
7,nov,4101,416,10.14%
8,jul,7174,649,9.05%
9,may,13769,886,6.43%



Day Of Week comparison:


,segment,records,responses,response_rate
0,thu,8623,1045,12.12%
1,tue,8090,953,11.78%
2,wed,8134,949,11.67%
3,fri,7827,846,10.81%
4,mon,8514,847,9.95%


## 5. Interactive cohort example

The dashboard supports all available category filters and numeric ranges. This cell demonstrates the same shared filter function with an example cohort.

In [8]:
# Example: cellular contacts, ages 30–60. Edit these values to explore another cohort.
example = apply_filters(data, {'contact': ['cellular']}, {'age': (30, 60)})
example_kpis = calculate_kpis(example)
print(f"Records: {example_kpis['records']:,}; successful responses: {example_kpis['responses']:,}; response rate: {example_kpis['response_rate']:.2%}")
display(example.head(10))

Records: 21,508; successful responses: 2,670; response rate: 12.41%

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
12357,45,blue-collar,married,basic.9y,no,yes,no,cellular,jul,fri,...,1,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12360,33,technician,married,basic.9y,Unknown,yes,no,cellular,jul,fri,...,4,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12361,41,management,married,university.degree,no,yes,no,cellular,jul,fri,...,3,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12363,30,services,divorced,high.school,no,yes,yes,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12364,58,retired,married,professional.course,Unknown,yes,no,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12365,31,blue-collar,married,basic.9y,no,Unknown,Unknown,cellular,jul,fri,...,3,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12366,30,admin.,married,high.school,no,yes,no,cellular,jul,fri,...,6,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12367,40,blue-collar,divorced,basic.6y,Unknown,Unknown,Unknown,cellular,jul,fri,...,3,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12368,33,services,married,high.school,no,yes,no,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12369,38,blue-collar,married,basic.9y,no,no,no,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no


## 6. Findings and recommendations

Findings require at least 30 records per segment. Recommendations remain hypotheses for testing; observed rates do not establish a causal effect.

In [9]:
for item in generate_findings(data):
    print(f"\n{item['title']}")
    print(f"Finding: {item['finding']}")
    print(f"Suggested next step: {item['recommendation']}")


Highest observed response: Job


Finding: student has the highest observed rate in the selected cohort: 275 of 875 records (31.4%). Groups below 30 records are omitted.
Suggested next step: Use this segment as a hypothesis for a targeted campaign test; this descriptive association does not establish that the segment caused the response.

Contact-frequency comparison
Finding: Response was 13.0% after one contact (17,642 records) and 6.1% after 5+ contacts (4,984 records); the 5+ band was 6.9 percentage points lower.
Suggested next step: Review the contact cadence for the one contact group before changing follow-up rules; the comparison is observational and may reflect differences in who was contacted.

Communication-channel comparison
Finding: cellular has the highest observed channel rate at 14.7% (3,853/26,144); the comparison channel telephone is 5.2% (787/15,044).
Suggested next step: Use channel differences to design a controlled follow-up test; customer mix and campaign timing can also explain the gap.


## 7. Limitations and interpretation

- `duration` is call duration in seconds and is only known after contact; it should not inform pre-call targeting.
- `pdays` uses the source dataset's encoding; 999 indicates the client was not previously contacted.
- A row is a contact record, not a verified unique customer.
- This is historical observational data. Patterns may reflect selection, customer mix, timing, and other confounders. Validate proposed campaign changes with controlled experiments.
- Source: Moro, S., Rita, P., & Cortez, P. (2014). Bank Marketing. UCI Machine Learning Repository. DOI: [10.24432/C5K306](https://doi.org/10.24432/C5K306). License: CC BY 4.0.